# Download Texurepacks/Dataset

fetch official versions list provided by modrinth

In [1]:
import json
import re

with open("versions.json", "r", encoding="utf-8") as file:
    versions = json.load(file)

release_pattern = re.compile(r"^\d+(\.\d+)*$")  # e.g. 1.20.1, 26.2, 1.21

releases = [entry.get("version") for entry in versions if release_pattern.match(entry.get("version"))]
print(releases)

['26.2', '26.1.2', '26.1.1', '26.1', '1.21.11', '1.21.10', '1.21.9', '1.21.8', '1.21.7', '1.21.6', '1.21.5', '1.21.4', '1.21.3', '1.21.2', '1.21.1', '1.21', '1.20.6', '1.20.5', '1.20.4', '1.20.3', '1.20.2', '1.20.1', '1.20', '1.19.4', '1.19.3', '1.19.2', '1.19.1', '1.19', '1.18.2', '1.18.1', '1.18', '1.17.1', '1.17', '1.16.5', '1.16.4', '1.16.3', '1.16.2', '1.16.1', '1.16', '1.15.2', '1.15.1', '1.15', '1.14.4', '1.14.3', '1.14.2', '1.14.1', '1.14', '1.13.2', '1.13.1', '1.13', '1.12.2', '1.12.1', '1.12', '1.11.2', '1.11.1', '1.11', '1.10.2', '1.10.1', '1.10', '1.9.4', '1.9.3', '1.9.2', '1.9.1', '1.9', '1.8.9', '1.8.8', '1.8.7', '1.8.6', '1.8.5', '1.8.4', '1.8.3', '1.8.2', '1.8.1', '1.8', '1.7.10', '1.7.9', '1.7.8', '1.7.7', '1.7.6', '1.7.5', '1.7.4', '1.7.3', '1.7.2', '1.7.1', '1.7', '1.6.4', '1.6.3', '1.6.2', '1.6.1', '1.6', '1.5.2', '1.5.1', '1.5', '1.4.7', '1.4.6', '1.4.5', '1.4.4', '1.4.3', '1.4.2', '1.4.1', '1.4', '1.3.2', '1.3.1', '1.3', '1.2.5', '1.2.4', '1.2.3', '1.2.2', '1.2.1'

In [2]:
import requests
import json

modrinth = "https://api.modrinth.com/v2/search"
hits = []

for r in releases:
    facets = [
        ["project_type:resourcepack"],
        ["categories:32x"],
        [f"versions:{r}"],
    ]
    params = {
        "query": "pvp",
        "facets": json.dumps(facets),
        "index": "downloads",
        "limit": 100,
    }
    req = requests.get(modrinth, params=params)
    req.raise_for_status()
    data = req.json()
    hits.append((r, data["total_hits"]))


find top 5 versions

In [3]:
hits = sorted(hits, key=lambda x: x[1], reverse=True)

for ver in hits[:5]:
    print(f"Version: {ver[0]}, Total Hits: {ver[1]}")

Version: 1.21.4, Total Hits: 73
Version: 1.21, Total Hits: 73
Version: 1.21.1, Total Hits: 72
Version: 1.21.3, Total Hits: 69
Version: 1.21.2, Total Hits: 68


In [7]:
params = {
    "query": "pvp",
    "facets": json.dumps([
        ["project_type:resourcepack"],
        ["categories:32x"],
        ["versions:1.21", "versions:1.21.1", "versions:1.21.2", "versions:1.21.3", "versions:1.21.4"],
    ]),
    "index": "downloads",
    "limit": 100,
}

req = requests.get(modrinth, params=params)
req.raise_for_status()
data = req.json()
project_ids = [hit["project_id"] for hit in data["hits"]]

In [17]:
import os
from dotenv import load_dotenv

load_dotenv()

data_dir = os.getenv("DATA_DIR")

In [ ]:
from tqdm import tqdm

os.makedirs(data_dir, exist_ok=True)

for project in tqdm(project_ids, desc="Projects", unit="proj"):
    versions = requests.get(f"https://api.modrinth.com/v2/project/{project}/version").json()
    if not versions:
        continue

    latest = max(versions, key=lambda v: v["date_published"])
    for f in latest["files"]:
        url = f["url"]
        filename = f["filename"]
        filepath = os.path.join(data_dir, filename)

        resp = requests.get(url, stream=True)
        resp.raise_for_status()
        total = int(resp.headers.get("content-length", 0))

        with open(filepath, "wb") as out, tqdm(
            total=total, unit="B", unit_scale=True, desc=filename, leave=False
        ) as file_bar:
            for chunk in resp.iter_content(chunk_size=8192):
                out.write(chunk)
                file_bar.update(len(chunk))

Downloaded: Vanilla+ 1.21.11.zip
Downloaded: better-low-fire-0.1.16+mc1.20-26.2.zip
Downloaded: Faithful 32x Daggers 1.0-1.21.6.zip
Downloaded: §8§ Zipp CPVP Pack.zip
Downloaded: !§bArkhalis.zip
Downloaded: Shadow Wolf 26.1.zip
Downloaded: Celesta v3.4.zip
Downloaded: golden knight.zip
Downloaded: JJ PvP 32x.zip
Downloaded: Lil's PVP Pack 1.14+.zip
Downloaded: blade-of-miquella.zip
Downloaded: Cool Font Finished.zip
Downloaded: Good_Days_Faithful_PvP.zip
Downloaded: §6Helpful PvP Pack.zip
Downloaded: Vanilla-EPR.zip
Downloaded: §8Black§6Hole§f32x-4.zip
Downloaded: PvP 32x lightblue.zip
Downloaded: ! §b§l Candy.zip
Downloaded: ! Typa Blue 32x 1.18.zip
Downloaded: YOGI +.zip
Downloaded: Malevolence 1.21.4 by Upwqrd.zip
Downloaded: Void 32x.zip
Downloaded: §4§lOrange 32x.zip
Downloaded: 小鲨鱼PVP材质包  1.1v 1.20.1.zip
Downloaded: !   §cMirai §dKuriyama §7[32x].zip
Downloaded: §5Hacker material.zip
Downloaded: Short Swords.zip
Downloaded: skibidi.zip
Downloaded: Phanta's 3D Katana 1.21.6.zip
Do

KeyboardInterrupt: 